In [8]:
# Marcin Oleszczuk
# Pomoce, pseudokody z polecenia + filmiki na Youtube + artykuły + ewentualne porady kolegów z grupy

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import chain, combinations
from IPython.display import display

In [2]:
print("Loading data")
loaded_data = []
with open('store_data.csv', 'r') as data:
    rows = data.readlines()

for row in rows:
    elements = row.split(',')
    for element in elements:
        loaded_data.append(elements)


print("Data loaded")

Loading data
Data loaded


In [3]:
# liczba transakcji w ktorych wystepuje item / calkowita liczbe transakcji
support = lambda item, set_of_freq, no_transactions: np.float(set_of_freq[item]) / no_transactions
confidence = lambda item, element, set_of_freq, no_transactions: support(item, set_of_freq, no_transactions) / support(element, set_of_freq, no_transactions)

def frequent_patterns(min_supp, transactions):
    # utworzenie zbioru jednoelementowego, po ktorym nastapi indeksowanie
    set_of_items = {frozenset([item]) for transaction in transactions for item in transaction}
    # częstość wystepowania
    set_of_freq = defaultdict(int)
    f_item = dict()
    # Sprawdzenie, ktore wzorce pasuja, czyli znalezienie wozrców częstych
    single_candidate_set = prune_candidates(set_of_items, set_of_freq, transactions, min_supp)
    candidates = single_candidate_set
    k = 2
    while len(candidates):
        f_item[k - 1] = candidates
        # wyznaczanie zbiorów kandydatów za pomoca techniki brute-force
        candidates = generate_candidates(candidates, k)
        candidates = prune_candidates(candidates, set_of_freq, transactions, min_supp)
        k = k + 1

    # wyznacznie wsparcia wzorców o duzej częstości
    items = []
    no_transactions = len(transactions)
    for key, value in f_item.items():
        items.extend([(tuple(item), support(item, set_of_freq, no_transactions)) for item in value])

    return f_item, set_of_freq, items


def generate_rules(f_item, set_of_freq, min_conf, transactions):
    rules = []
    no_transactions = len(transactions)
    # Wyznaczanie reguł dla zbierów posiadających więcej niz 1 element
    for key, value in list(f_item.items())[1:]:
        for item in value:
            # Obliczanie poprzedników
            previous_candidates = map(frozenset, [x for x in chain(*[combinations(item, i + 1) for i, a in enumerate(item)])])
            for element in previous_candidates:
                remain = item.difference(element)
                if len(remain) > 0:
                    el_confidence = confidence(item, element, set_of_freq, no_transactions)
                    if el_confidence >= min_conf:
                        rules.append(((tuple(element), tuple(remain)), el_confidence))

    # Sortowanie reguł po confidence
    rules = sort_rules(set_of_freq, rules, transactions)
    return rules

def add_support_to_rule(rules, set_of_freq, no_transactions):
    sorted_rules = []
    for rule in rules:
        (element, remain), el_confidence = rule
        element = frozenset(element)
        remain = frozenset(remain)
        item = element.union(remain)
        rule_support = support(item, set_of_freq, no_transactions)
        sorted_rules.append(((tuple(element), tuple(remain)), el_confidence, rule_support))
    return sorted_rules

def sort_rules(set_of_freq, rules, transactions):
    sorted_rules = add_support_to_rule(rules, set_of_freq, len(transactions))
    sorted_rules = sorted(sorted_rules, key=lambda x: x[1], reverse=True)
    return sorted_rules


def generate_candidates(candidates, k):
    # Generowanie kandydatów poprzez wytworzenie wszystkich mozliwych połączeń miedzy itemami
    return set([i.union(j) for i in candidates for j in candidates if len(i.union(j)) == k])

def count_candidates(candidates, set_of_freq, transactions):
    candidate_set = defaultdict(int)
    for item in candidates:
        for transaction in transactions:
            if item.issubset(transaction):
                set_of_freq[item] += 1
                candidate_set[item] += 1

    return candidate_set

def prune_candidates(candidates, set_of_freq, transactions, min_supp):
    result = set()
    candidate_set = count_candidates(candidates, set_of_freq, transactions)

    for item, count in candidate_set.items():
        supp = float(count) / len(transactions)

        if supp >= min_supp:
            result.add(item)

    return result

def print_results(items, rules):
    to_add = []
    for item, item_support in sorted(items, key=lambda x: x[1], reverse=True):
        df_elem = {'Pattern': '->'.join(item), 'Support': f'{item_support:.4f}'}
        to_add.append(df_elem)
    pattern_df = pd.DataFrame(to_add)
    display("Patterns")
    display(pattern_df)

    rule_display = []
    for rule, el_confidence, rule_support in rules:
        pre, post = rule
        rule_display.append({
            'If': '->'.join(pre), 'Then': '->'.join(post), 'Confidence': f'{el_confidence:.3f}', 'Support': f'{rule_support:.3f}'
        })

    rule_df = pd.DataFrame(rule_display)
    display("Rules")
    display(rule_df)

In [4]:
min_supp=0.05
min_conf=0.1
freq_item, set_of_freq, items = frequent_patterns(min_supp, loaded_data)
rules = generate_rules(freq_item, set_of_freq, min_conf, loaded_data)

In [5]:
print_results(items, rules)

'Patterns'

,Pattern,Support
0,mineral water,0.3438
1,spaghetti,0.2754
2,chocolate,0.2150
3,eggs,0.2134
4,milk,0.2051
5,french fries,0.1731
6,ground beef,0.1674
7,frozen vegetables,0.1592
8,green tea\n,0.1402
9,burgers,0.1368


'Rules'

,If,Then,Confidence,Support
0,olive oil,mineral water,0.489,0.053
1,ground beef,mineral water,0.471,0.079
2,ground beef,spaghetti,0.468,0.078
3,milk,mineral water,0.463,0.095
4,pancakes,mineral water,0.449,0.058
5,frozen vegetables,mineral water,0.444,0.071
6,spaghetti,mineral water,0.427,0.118
7,chocolate,mineral water,0.407,0.087
8,pancakes,spaghetti,0.402,0.052
9,eggs,mineral water,0.394,0.084
